# Agentic API Testing & Documentation Generator 🧪🤖

A portfolio starter that reads an OpenAPI specification, discovers endpoints, creates test scenarios, runs a safe in-memory mock test suite, and generates Markdown API documentation.

**Safety:** The default runner tests only an in-memory mock API. Test only systems you own or are authorized to assess.

## Features
- OpenAPI 3.x JSON/YAML loading
- Endpoint and schema discovery
- Positive, negative, boundary test-case generation
- Optional LLM test suggestions
- Safe mock API test execution
- Expected-versus-actual result reporting
- Markdown API documentation generation
- CSV and JSON exports

In [ ]:
%pip install -q pandas pyyaml jsonschema python-dotenv openai

In [ ]:
import os, json, re
from pathlib import Path
import pandas as pd
import yaml

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
API_KEY=os.getenv("OPENAI_API_KEY","")
BASE_URL=os.getenv("OPENAI_BASE_URL","").strip() or None
MODEL=os.getenv("OPENAI_MODEL","gpt-4.1-mini")
print("LLM configured:",bool(API_KEY))

## 1. Sample OpenAPI specification

In [ ]:
spec={
 "openapi":"3.0.3",
 "info":{"title":"Demo Task API","version":"1.0.0","description":"Example task management API."},
 "paths":{
  "/tasks":{
   "get":{"summary":"List tasks","parameters":[{"name":"status","in":"query","required":False,"schema":{"type":"string","enum":["open","done"]}}],"responses":{"200":{"description":"Task list"}}},
   "post":{"summary":"Create a task","requestBody":{"required":True,"content":{"application/json":{"schema":{"type":"object","required":["title"],"properties":{"title":{"type":"string","minLength":1,"maxLength":100},"priority":{"type":"integer","minimum":1,"maximum":5}}}}}},"responses":{"201":{"description":"Created"},"400":{"description":"Invalid input"}}}
  },
  "/tasks/{task_id}":{
   "get":{"summary":"Get a task","parameters":[{"name":"task_id","in":"path","required":True,"schema":{"type":"integer","minimum":1}}],"responses":{"200":{"description":"Found"},"404":{"description":"Not found"}}},
   "delete":{"summary":"Delete a task","parameters":[{"name":"task_id","in":"path","required":True,"schema":{"type":"integer","minimum":1}}],"responses":{"204":{"description":"Deleted"},"404":{"description":"Not found"}}}
  }
 }
}
SPEC_PATH="" # Optional path to your OpenAPI .json/.yaml file
def load_spec(path):
    raw=Path(path).read_text(encoding="utf-8")
    return yaml.safe_load(raw) if Path(path).suffix.lower() in {".yaml",".yml"} else json.loads(raw)
if SPEC_PATH: spec=load_spec(SPEC_PATH)
print(spec["info"])

## 2. Discover endpoints

In [ ]:
METHODS={"get","post","put","patch","delete","options","head"}
def discover(spec):
    found=[]
    for path,item in spec.get("paths",{}).items():
        for method,op in item.items():
            if method.lower() in METHODS and isinstance(op,dict):
                found.append({"method":method.upper(),"path":path,"summary":op.get("summary",""),
                              "parameters":op.get("parameters",[]),"request_body":op.get("requestBody"),
                              "responses":op.get("responses",{})})
    return found
endpoints=discover(spec)
display(pd.DataFrame([{"method":e["method"],"path":e["path"],"summary":e["summary"],
                       "responses":", ".join(e["responses"].keys())} for e in endpoints]))

## 3. Generate test scenarios from the API contract

In [ ]:
def make_cases(endpoint):
    method,path=endpoint["method"],endpoint["path"]
    cases=[{"method":method,"path":path,"category":"positive",
            "scenario":"Valid request based on the endpoint contract","priority":"high"}]
    for param in endpoint["parameters"]:
        cases.append({"method":method,"path":path,"category":"boundary",
                      "scenario":f"Test valid and boundary values for {param['name']}","priority":"medium"})
        if param.get("required"):
            cases.append({"method":method,"path":path,"category":"negative",
                          "scenario":f"Omit required parameter {param['name']}","priority":"high"})
    if endpoint["request_body"]:
        cases += [
          {"method":method,"path":path,"category":"positive","scenario":"Submit a valid request body","priority":"high"},
          {"method":method,"path":path,"category":"negative","scenario":"Submit missing/invalid required fields","priority":"high"}
        ]
    if "{" in path:
        cases.append({"method":method,"path":path,"category":"negative",
                      "scenario":"Request a non-existent resource","priority":"medium"})
    return cases
cases=[c for e in endpoints for c in make_cases(e)]
case_df=pd.DataFrame(cases).drop_duplicates()
display(case_df)

## 4. Optional LLM-assisted test suggestions
Treat generated suggestions as ideas; review them before execution.

In [ ]:
def suggest_with_llm(endpoint_summary):
    if not API_KEY: raise RuntimeError("Set OPENAI_API_KEY to enable LLM suggestions.")
    from openai import OpenAI
    args={"api_key":API_KEY}
    if BASE_URL: args["base_url"]=BASE_URL
    client=OpenAI(**args)
    result=client.chat.completions.create(
      model=MODEL,temperature=0.2,response_format={"type":"json_object"},
      messages=[
       {"role":"system","content":"You are an API QA engineer. Return JSON with a test_suggestions array. Suggest safe positive, negative, boundary, and authorization test ideas. Do not suggest destructive or unauthorized actions."},
       {"role":"user","content":json.dumps(endpoint_summary)}
      ])
    return json.loads(result.choices[0].message.content or "{}")
llm_ideas=None
if API_KEY:
    try:
        llm_ideas=suggest_with_llm([{"method":e["method"],"path":e["path"],"summary":e["summary"]} for e in endpoints])
        display(pd.DataFrame(llm_ideas.get("test_suggestions",[])))
    except Exception as exc: print("LLM suggestions unavailable:",exc)
else: print("No API key: deterministic test generation is active.")

## 5. Execute safe tests against an in-memory mock API
This demonstration deliberately sends no requests to external systems.

In [ ]:
class MockTaskAPI:
    def __init__(self):
        self.tasks={1:{"id":1,"title":"Prepare release notes","priority":2,"status":"open"}}
        self.next_id=2
    def handle(self,method,path,params=None,body=None):
        params,body=params or {},body or {}
        if path=="/tasks" and method=="GET":
            status=params.get("status")
            return 200,[v for v in self.tasks.values() if status is None or v["status"]==status]
        if path=="/tasks" and method=="POST":
            title=body.get("title"); priority=body.get("priority",3)
            if not isinstance(title,str) or not title.strip() or len(title)>100: return 400,{"error":"invalid title"}
            if not isinstance(priority,int) or not 1<=priority<=5: return 400,{"error":"invalid priority"}
            task={"id":self.next_id,"title":title,"priority":priority,"status":"open"}
            self.tasks[self.next_id]=task; self.next_id+=1
            return 201,task
        match=re.fullmatch(r"/tasks/(\\d+)",path)
        if match:
            task_id=int(match.group(1))
            if method=="GET": return (200,self.tasks[task_id]) if task_id in self.tasks else (404,{"error":"not found"})
            if method=="DELETE":
                if task_id not in self.tasks: return 404,{"error":"not found"}
                del self.tasks[task_id]; return 204,None
        return 404,{"error":"route not found"}

def run_tests(api):
    tests=[
      ("list_tasks","GET","/tasks",{}, {},200),
      ("create_valid","POST","/tasks",{},{"title":"Write API tests","priority":4},201),
      ("create_missing_title","POST","/tasks",{},{"priority":2},400),
      ("create_bad_priority","POST","/tasks",{},{"title":"Invalid","priority":9},400),
      ("get_existing","GET","/tasks/1",{}, {},200),
      ("get_missing","GET","/tasks/999",{}, {},404),
      ("delete_existing","DELETE","/tasks/1",{}, {},204),
      ("delete_missing","DELETE","/tasks/999",{}, {},404)]
    rows=[]
    for name,method,path,params,body,expected in tests:
        actual,payload=api.handle(method,path,params,body)
        rows.append({"test":name,"method":method,"path":path,"expected":expected,
                     "actual":actual,"passed":actual==expected,"response":payload})
    return pd.DataFrame(rows)
results=run_tests(MockTaskAPI())
display(results)
print("Passed:",int(results.passed.sum()),"/",len(results))

## 6. Generate Markdown API documentation

In [ ]:
def build_docs(spec,endpoints):
    info=spec.get("info",{})
    lines=[f"# {info.get('title','API Documentation')}","",info.get("description",""),"",
           f"**OpenAPI:** {spec.get('openapi','unknown')}  ",
           f"**API version:** {info.get('version','unknown')}","",
           "## Endpoints"]
    for e in endpoints:
        lines += ["",f"### `{e['method']} {e['path']}`",e["summary"] or "No summary provided.",""]
        if e["parameters"]:
            lines += ["| Parameter | Location | Required | Schema |","|---|---|---:|---|"]
            for x in e["parameters"]:
                lines.append(f"| {x.get('name')} | {x.get('in')} | {x.get('required',False)} | `{json.dumps(x.get('schema',{}))}` |")
            lines.append("")
        lines += ["**Responses**","", "| Status | Description |","|---|---|"]
        for status,response in e["responses"].items():
            lines.append(f"| {status} | {response.get('description','')} |")
    return "\\n".join(lines)
docs=build_docs(spec,endpoints)
print(docs)

## 7. Export test cases, results, and documentation

In [ ]:
out=Path("api_test_output"); out.mkdir(exist_ok=True)
case_df.to_csv(out/"generated_test_cases.csv",index=False)
results.to_csv(out/"test_results.csv",index=False)
(out/"test_report.json").write_text(results.to_json(orient="records",indent=2),encoding="utf-8")
(out/"api_documentation.md").write_text(docs,encoding="utf-8")
print("Files exported to:",out.resolve())

## 8. Future enhancements
- Resolve OpenAPI `$ref` and advanced schema features.
- Generate executable pytest modules and response-schema assertions.
- Add authorized staging API execution with timeouts and rate limits.
- Add contract testing, mock servers, and CI integration.
- Add failure clustering, test coverage analytics, and LLM evaluation.
- Require human approval for state-changing tests.

**Responsible use:** Only test APIs you own or are explicitly authorized to assess. Keep destructive tests disabled by default.